# Tarea 07: Una cuenta en un aro que gira

**Semana 7** · Entrega: antes de la clase de la Semana 8

## Objetivos de la tarea

1. Orientar marcos de referencia encadenados, con ángulos que dependen del
   tiempo, y leer sus matrices de cosenos directores.
2. Obtener la posición, velocidad y aceleración de un punto con `Point`, y
   verificarlas con el teorema de dos puntos.
3. Interpretar físicamente los términos de la rapidez y de la aceleración.

In [1]:
import sympy as sp
import sympy.physics.mechanics as me
from IPython.display import Math

me.init_vprinting()

# Ya declarados: no los cambies, los tests los usan tal cual.
t = sp.Symbol("t")               # el tiempo de mechanics: sin suposiciones
N = me.ReferenceFrame("N")       # el marco del laboratorio
centro = me.Point("O")           # el centro del aro
centro.set_vel(N, 0)             # fijo en el laboratorio

## El sistema

Un aro de alambre de radio $R$ está parado verticalmente y **gira** alrededor
de su diámetro vertical con velocidad angular constante $\Omega$. Una cuenta
ensartada en el alambre puede deslizarse por él; su posición sobre el aro la da
el ángulo $\theta(t)$, medido desde el punto más bajo.

```
            n_z
             |
          .--+--.
        /    |    \
       |     O     |     ← el aro gira alrededor de n_z con Ω
        \    |  θ /
          '--+--●         ● la cuenta
             |
```

Lo describimos con tres marcos encadenados:

- `N`: el laboratorio, con $\hat{n}_z$ vertical hacia arriba.
- `A`: el **aro**. Es $N$ girado un ángulo $\Omega t$ alrededor de `N.z`, así
  que el plano del aro es el plano $\hat{a}_x$–$\hat{a}_z$.
- `B`: la **cuenta**. Es $A$ girado un ángulo $\theta$ alrededor de `A.y`, el
  eje perpendicular al plano del aro.

En este arreglo la cuenta está en $-R\,\hat{b}_z$: en $\theta = 0$ eso es el
punto más bajo del aro, justo debajo del centro.

Todo lo que necesitas está en los dos notebooks de la semana 7.

## Ejercicio 1: símbolos y marcos

Completa la celda con estos nombres exactos, que son los que usa
`tests/test_tarea.py`:

- `radio` y `omega`: símbolos **positivos**, con nombres de símbolo `"R"` y
  `"Omega"` — un radio y una rapidez de giro no son negativos.
- `theta`: el ángulo de la cuenta, como `dynamicsymbol` de nombre `"theta"`.
- `A` y `B`: los marcos del aro y de la cuenta, orientados como dice la
  descripción del sistema, con `orient_axis`.

Antes de seguir, muestra `A.dcm(N)` y `B.dcm(A)` y compáralos con la DCM de la
sesión 1. Una de las dos no tiene la misma forma que la de la clase: ¿por qué?

In [2]:
# TODO: completa el ejercicio 1
radio = sp.Symbol("R", positive=True)
omega = sp.Symbol("Omega", positive=True)
theta = me.dynamicsymbols("theta")

A = me.ReferenceFrame("A")
# A es N girado un ángulo omega*t alrededor del eje N.z
A.orient_axis(N, N.z, omega * t)

B = me.ReferenceFrame("B")
# B es A girado un ángulo theta alrededor del eje A.y
B.orient_axis(A, A.y, theta)

# Mostrar las matrices de cosenos directores (DCM)
display(Math(r"^{A}C^{N}:"))
display(A.dcm(N))

display(Math(r"^{B}C^{A}:"))
display(B.dcm(A))

<IPython.core.display.Math object>

⎡cos(Ω⋅t)   sin(Ω⋅t)  0⎤
⎢                      ⎥
⎢-sin(Ω⋅t)  cos(Ω⋅t)  0⎥
⎢                      ⎥
⎣    0         0      1⎦

<IPython.core.display.Math object>

⎡cos(θ)  0  -sin(θ)⎤
⎢                  ⎥
⎢  0     1     0   ⎥
⎢                  ⎥
⎣sin(θ)  0  cos(θ) ⎦

La matriz A.dcm(N) corresponde a una rotación alrededor del eje $Z$, la cual mezcla las coordenadas $X$ y $Y$ (y tiene su bloque principal en la esquina superior izquierda). En cambio, la matriz B.dcm(A) representa una rotación alrededor del eje $Y$, afectando las coordenadas $X$ y $Z$.

## Ejercicio 2: posición, velocidad y aceleración

- `cuenta`: un `me.Point` ubicado respecto a `centro` en $-R\,\hat{b}_z$.
- `posicion`: la posición de la cuenta respecto a `centro`.
- `velocidad` y `aceleracion`: su velocidad y su aceleración **en `N`**.

Las expresiones en `N` salen largas. Para leerlas, muéstralas con `express(B)`,
en la base de la cuenta: se simplifican mucho. Etiquétalas con `Math` y
`me.vlatex`, como en clase.

In [3]:
# TODO: completa el ejercicio 2
cuenta = me.Point("cuenta")

# La posición de la cuenta respecto al centro
posicion = -radio * B.z
cuenta.set_pos(centro, posicion)

# Velocidad y aceleración en el marco del laboratorio (N)
velocidad = posicion.dt(N)
aceleracion = velocidad.dt(N)
cuenta.set_vel(N, velocidad)
cuenta.set_acc(N, aceleracion)

# Mostrar las expresiones en la base B etiquetadas con Math y vlatex
display(Math(r"\mathbf{r} = " + me.vlatex(posicion.express(B))))
display(Math(r"\mathbf{v} = " + me.vlatex(velocidad.express(B).simplify())))
display(Math(r"\mathbf{a} = " + me.vlatex(aceleracion.express(B).simplify())))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Ejercicio 3: velocidad angular y teorema de transporte

- `velocidad_angular`: la velocidad angular de `B` en `N`. Debe tener dos
  pedazos: el giro del aro y el deslizamiento de la cuenta.
- `velocidad_transporte`: $\boldsymbol{\omega}^B \times \mathbf{r}$, el producto
  cruz de `velocidad_angular` con `posicion`.

`velocidad_transporte` tiene que coincidir con la `velocidad` del ejercicio 2.
Compruébalo tú: la diferencia, expresada en `N` y simplificada, debe ser cero.
¿Qué teorema de la sesión 2 estás usando, y por qué aplica aquí aunque la
cuenta **sí** se mueve sobre el aro?

In [4]:
# TODO: completa el ejercicio 3
velocidad_angular = B.ang_vel_in(N)

velocidad_transporte = me.cross(velocidad_angular, posicion)

# Comprobación: la diferencia debe ser el vector nulo (cero)
diferencia = (velocidad - velocidad_transporte).express(N).simplify()
print("¿Coinciden la velocidad y la de transporte?:", diferencia == 0)

¿Coinciden la velocidad y la de transporte?: True


## Ejercicio 4: rapidez y aceleración a lo largo del aro

- `rapidez_cuadrada`: $\mathbf{v} \cdot \mathbf{v}$, simplificada con
  `sp.simplify`. Es un escalar: etiquétalo con `sp.Eq`.
- `aceleracion_tangencial`: la componente de `aceleracion` a lo largo de
  $\hat{b}_x$, simplificada.

Todo con cantidades exactas: nada de decimales.

In [5]:
# TODO: completa el ejercicio 4
rapidez_cuadrada = sp.simplify(me.dot(velocidad, velocidad))

aceleracion_tangencial = sp.simplify(me.dot(aceleracion, B.x))

# Mostrar los resultados exactos
display(sp.Eq(sp.Symbol("v^2"), rapidez_cuadrada))
display(Math(r"a_t = " + sp.latex(aceleracion_tangencial)))

<IPython.core.display.Math object>

## Discusión

_Contesta aquí, en uno o dos párrafos. Esta parte no la califica el
autograding: la revisa el asistente._

1. `rapidez_cuadrada` tiene dos términos. ¿Cuál viene de que la cuenta se
   desliza por el aro y cuál de que el aro gira? ¿Por qué el segundo se anula en
   $\theta = 0$?
2. En `aceleracion_tangencial` aparece $-R\ddot\theta$, con signo menos. ¿Por
   qué? (Pista: ¿hacia dónde apunta $\hat{b}_x$ cuando $\theta$ crece?)
3. El término con $\Omega^2$ de `aceleracion_tangencial` es el que, visto desde
   el aro, se siente como la "fuerza centrífuga" que empuja a la cuenta hacia
   afuera. ¿En qué valores de $\theta$ se anula, y por qué tiene sentido
   físicamente en cada uno?

_(Escribe tu respuesta aquí.)_

1. En la `rapidez_cuadrada`, el término que contiene $\dot{\theta}^2$ representa la energía cinética del deslizamiento de la cuenta a lo largo del aro, mientras que el término con $\Omega^2$ proviene de la velocidad de arrastre por el giro del aro. Este último se anula en $\theta = 0$ porque en el punto más bajo la cuenta se encuentra exactamente sobre el eje de rotación vertical; al no tener distancia perpendicular al eje, su radio de giro efectivo y su velocidad tangencial de arrastre son cero.

2. En la `aceleracion_tangencial`, el término de la aceleración angular aparece como $-R\ddot{\theta}$ debido a la geometría del marco rotado: al definir la posición en $-R\hat{b}_z$ y rotar en $\hat{a}_y$, la cinemática dicta que el vector $\hat{b}_x$ apunte en dirección contraria al movimiento cuando $\theta$ crece. 

3. El término de la fuerza centrífuga (proporcional a $\Omega^2$) se anula en $\theta = 0$ y $\pi$ (donde la cuenta está sobre el eje y no hay fuerza centrífuga) y en $\theta = \pm \pi/2$ (donde la fuerza centrífuga es máxima, pero es completamente horizontal y perpendicular al alambre, por lo que su proyección o componente tangencial a lo largo del aro es nula).

## Resumen

Entrega esta tarea por **Pull Request dentro de tu fork**, como las anteriores.
El paso a paso está en [`docs/git-guia.md`](../../docs/git-guia.md), y el
checklist completo en [`README.md`](README.md).

Antes de entregar, corre el notebook desde un kernel limpio
(`Kernel → Restart & Run All`) y confirma que llega hasta el final sin errores.
Al hacer push, GitHub Actions corre los tests y deja un check ✅ o ❌ en tu
Pull Request.